In [1]:
import os
import sys
import numpy as np
import pandas as pd 
import gensim
from gensim import corpora
if '/mnt/ssd1/encore/open-source' not in sys.path: sys.path.insert(0, '/mnt/ssd1/encore/open-source')
from utils.initialization import *
from utils.distributions import *
from utils.eval import *

ModuleNotFoundError: No module named 'utils.distributions'

In [ ]:
os.chdir('/mnt/ssd1/encore/open-source')
app = 'app_182'
size_dir = './data/size/'
interval_dir = './data/interval/'
metadata_dir = './data/metadata/'
size_cdf = pd.read_csv('./data/cdf/size_cdf.csv')
interval_cdf = pd.read_csv('./data/cdf/interval_cdf.csv')
n_size = len(size_cdf) - 1
n_interval = len(interval_cdf) - 1
file = app + '.txt'
block_size = 30
batch_size = 64

In [ ]:
def get_dists(data: list) -> tuple:
    size_dists, interval_dists = [], []
    for seq in data:
        seq = np.append(seq[:-1], seq[0:block_size-1])
        size_seq = seq // n_interval
        interval_seq = seq % n_interval
        size_dists.append(compute_probability_distribution(size_seq, n_size))
        interval_dists.append(compute_probability_distribution(interval_seq, n_interval))
    return np.array(size_dists), np.array(interval_dists)

data = get_data(size_dir, interval_dir, file, n_interval)
size_dists, interval_dists = get_dists(data)

In [ ]:
model_dir = './checkpoints/lomas'
dict_file = os.path.join(model_dir, 'dictionary', app + '.dict')
topic_probs_file = os.path.join(model_dir, 'topic_probs', app + '.txt')
topic_word_probs_file = os.path.join(model_dir, 'topic_word_probs', app + '.txt')
dictionary = corpora.Dictionary.load(dict_file)
topic_probs = np.loadtxt(topic_probs_file, delimiter=',')
topic_word_probs = np.loadtxt(topic_word_probs_file, delimiter=',')
topic_word_probs = topic_word_probs / topic_word_probs.sum(axis=1)[:, np.newaxis]

In [ ]:
word_prob = np.dot(topic_probs, topic_word_probs)

In [ ]:
size_lomas, interval_lomas = [], []
num_sample = topic_probs.shape[0]
num_topics = topic_probs.shape[1]
word_prob = np.dot(topic_probs, topic_word_probs)
word_prob = word_prob / word_prob.sum(axis=1)[:, np.newaxis]
word_prob_matrix = word_prob.reshape(num_sample, n_size, n_interval)
size_prob = np.sum(word_prob_matrix, axis=2)
interval_prob = np.sum(word_prob_matrix, axis=1)

lomas_errors = {}
lomas_errors['size_dist'] = cramer_dis_matrix(size_dists, size_prob)
lomas_errors['interval_dist'] = cramer_dis_matrix(interval_dists, interval_prob)
lomas_errors['total_dist'] = lomas_errors['size_dist'] + lomas_errors['interval_dist']

base_path = './results/lomas_{file}'
for key, value in lomas_errors.items():
    directory = base_path.format(file=key)
    os.makedirs(directory, exist_ok=True)
    result_file = f'{directory}/{app}.txt'
    np.savetxt(result_file, value, delimiter=',', fmt='%.3f')

In [ ]:
def compute_size_interval_jsd(seq, seq_gen, n_interval):
    size_seq = np.array(seq) // n_interval
    interval_seq = np.array(seq) % n_interval
    size_seq_gen = np.array(seq_gen) // n_interval
    interval_seq_gen = np.array(seq_gen) % n_interval

    # Calculate original, he (shuffled), and gen (generated) distributions
    ori_pairs = list(zip(size_seq, interval_seq))
    permuted_interval_seq = np.random.permutation(interval_seq)
    permuted_pairs = list(zip(size_seq, permuted_interval_seq))
    gen_pairs = list(zip(size_seq_gen, interval_seq_gen))

    # Calculate distributions
    ori_dist = {pair: ori_pairs.count(pair) / len(ori_pairs) for pair in set(ori_pairs)}
    permuted_dist = {pair: permuted_pairs.count(pair) / len(permuted_pairs) for pair in set(permuted_pairs)}
    gen_dist = {pair: gen_pairs.count(pair) / len(gen_pairs) for pair in set(gen_pairs)}

    # Calculate JSD
    he_jsd = compute_js_divergence(ori_dist, permuted_dist)
    gen_jsd = compute_js_divergence(ori_dist, gen_dist)

    return he_jsd, gen_jsd


def test_gru(seqs, word_prob, n_interval, seq_len=1000, initial_seed=42):
    jsds = {
        'gen_size_size': {n: [] for n in [2, 3, 4]},
        'gen_size_interval': [],
    }
    for i, seq in enumerate(seqs):
        size_seq = np.array(seq) // n_interval
        seq_gen = np.random.choice(n_interval * n_size, seq_len, p=word_prob[i])
        size_seq_gen = np.array(seq_gen) // n_interval
        he_jsd, gen_jsd = compute_size_interval_jsd(seq, seq_gen, n_interval)
        jsds['gen_size_interval'].append(gen_jsd)
        for n in [2, 3, 4]:
            n_gram_ori = compute_ngram_distribution(size_seq, n)
            n_gram_gen = compute_ngram_distribution(size_seq_gen, n)
            jsds['gen_size_size'][n].append(compute_js_divergence(n_gram_ori, n_gram_gen))
    return jsds


gru_jsds = test_gru(data, word_prob, n_interval)
df = pd.DataFrame()
for key, value in gru_jsds.items():
    if isinstance(value, dict):
        for n, jsd in value.items():
            df[f'{key}_{n}'] = jsd
    else:
        df[key] = value
directory = './results/lomas_jsds'
os.makedirs(directory, exist_ok=True)
df.to_csv(f'{directory}/{app}.csv', index=False, float_format='%.4f')

{'gen_size_size': {2: [0.4907326054502495,
   0.6130217669187246,
   0.5590923218316154,
   0.5448855782221251,
   0.6246673634315585,
   0.5441862947346582,
   0.5680377684944835,
   0.55362080680518,
   0.6642485747609944,
   0.6656596889461461,
   0.521091052301307,
   0.3954617084292562,
   0.6069569781858675,
   0.492861677522156,
   0.5480929816642123,
   0.46355653671981806,
   0.47795356008592527,
   0.5388415422211899,
   0.5037197675069479,
   0.47968795601674613,
   0.5030207889543133,
   0.45059547414570106,
   0.5781679620835692,
   0.5111373589751894,
   0.6568155316670632,
   0.5953024988305484,
   0.5824573325826775,
   0.5197628411305177,
   0.5322091248417331,
   0.607648126485945,
   0.480185175359701,
   0.6496200795129874,
   0.5726780664541121,
   0.502051054393484,
   0.5190991377267712,
   0.6252543417948877,
   0.5183223714843221,
   0.6208330154550732,
   0.4812770885070686,
   0.4832552602007258,
   0.49797720153470054,
   0.5215413195617336,
   0.49314220264

In [ ]:
size_lomas, interval_lomas = [], []
for i in range(len(data)):
    topic_prob = np.zeros(32)
    for key, value in lda_model.get_document_topics(corpus[i], minimum_probability=None):
        topic_prob[key] = value
    topic_prob = topic_prob / sum(topic_prob)
    topic_word_prob = np.zeros(n_size * n_interval)

    for topic, prob in enumerate(topic_prob):
        for key, value in lda_model.show_topic(topic, topn=None):
            topic_word_prob[int(key)] += prob * value
    topic_word_prob = topic_word_prob / sum(topic_word_prob)

    word_prob = np.outer(topic_prob, topic_word_prob).sum(axis=0)
    word_prob_matrix = word_prob.reshape(n_size, n_interval)
    size_prob = np.sum(word_prob_matrix, axis=1)
    interval_prob = np.sum(word_prob_matrix, axis=0)
    size_lomas.append(size_prob)
    interval_lomas.append(interval_prob)
size_lomas = np.array(size_lomas)
interval_lomas = np.array(interval_lomas)

In [ ]:
lomas_error = cramer_dis_matrix(size_prob, size_dists)
all_data = np.concatenate(data)
all_sizes = all_data // n_interval
size_dist = compute_probability_distribution(all_sizes, n_size)
common_error = cramer_dis_matrix(np.reshape(size_dist, (1, n_size)), size_dists)
encore_error = np.loadtxt('./results/cvae_size_dist/app_182.txt', delimiter=',')

In [ ]:
def print_acc_cov(error):
    accuracy = np.sort(error.min(axis=1))
    coverage = np.sort(error.min(axis=0))
    acc_percentile = np.percentile(accuracy, [50, 75, 90, 95, 99])
    cov_percentile = np.percentile(coverage, [50, 75, 90, 95, 99])
    print('accuracy: {:.2e}/{:.2e}/{:.2e}/{:.2e}/{:.2e}'.format(*acc_percentile))
    print('coverage: {:.2e}/{:.2e}/{:.2e}/{:.2e}/{:.2e}'.format(*cov_percentile))

print('Lomas error')
print_acc_cov(lomas_error)
print('Common error')
print_acc_cov(common_error)
print('Encore error')
print_acc_cov(encore_error)

Lomas error
accuracy: 8.78e-03/2.64e-02/3.92e-02/4.51e-02/5.15e-02
coverage: 1.10e-02/1.42e-02/2.08e-02/2.39e-02/2.92e-02
Common error
accuracy: 1.53e-02/1.53e-02/1.53e-02/1.53e-02/1.53e-02
coverage: 3.17e-02/4.06e-02/1.19e-01/1.27e-01/1.38e-01
Encore error
accuracy: 8.00e-03/1.05e-02/1.20e-02/1.33e-02/1.50e-02
coverage: 8.00e-03/1.10e-02/1.30e-02/1.53e-02/1.79e-02


In [ ]:
topic_probs = []
for i in range(len(data)):
    topic_prob = np.zeros(32)
    for key, value in lda_model.get_document_topics(corpus[i], minimum_probability=None):
        topic_prob[key] = value
    topic_probs.append(topic_prob / sum(topic_prob))

topic_word_probs = []
for topic in range(32):
    topic_word_prob = np.zeros(n_size * n_interval)
    for key, value in lda_model.show_topic(topic, topn=None):
        topic_word_prob[int(key)] = value
    topic_word_probs.append(topic_word_prob / sum(topic_word_prob))
    
topic_probs = np.array(topic_probs)
topic_word_probs = np.array(topic_word_probs)
lomas_dir = './checkpoints/lomas/'
os.makedirs(os.path.join(lomas_dir, 'topic_probs'), exist_ok=True)
os.makedirs(os.path.join(lomas_dir, 'topic_word_probs'), exist_ok=True)
os.makedirs(os.path.join(lomas_dir, 'dictionary'), exist_ok=True)
np.savetxt(os.path.join(lomas_dir, 'topic_probs', app + '.txt'), topic_probs, delimiter=',', fmt='%.3f')
np.savetxt(os.path.join(lomas_dir, 'topic_word_probs', app + '.txt'), topic_word_probs, delimiter=',', fmt='%.3f')
dictionary.save(os.path.join(lomas_dir, 'dictionary', app + '.dict'))